# Test Functionality of GestionOt{class}

### Pasos para calificar una actividad.


1. Separar los eventos que tienen alimentador de los que no.
   
   1.1. Separar y calificar aquellos que son de TRANSPORTE, ALIMENTACIÓN, SE LABORA, INFO, se repite en la calificación
   
2. A los eventos que si tienen alimentador.
   
   2.1. Separar aquellos que sabemos que son SAPG, los más fáciles de identificar.

   2.2. Separar aquellos que son de Servicios Ocasionales.

   2.3. Calificar usando la Red Neuronal.

In [1]:

from eerssa import gestionOT
from eerssa import matrizActividades
from pathlib import Path
import pandas as pd

import pickle


#test_path = '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/'
test_path = '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/find_bug'
#test_path = '/home/vlad/OneDrive/01 JEZO/01 ACTIVIDADES DIARIAS DE TRABAJO DE LAS AGENCIAS/2024'
#test_path = '/home/vlad/OneDrive/01 JEZO/01 ACTIVIDADES DIARIAS DE TRABAJO DE LAS AGENCIAS/2025/02 FEBRERO'


save_dir = '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/db_test/'

file_prefix = '2025-02_Febrero'

path_obj = save_dir + file_prefix + '_0_object.pkl'
path_pkl = save_dir + file_prefix + '_0_df.pkl'
path_xls = save_dir + file_prefix + '_0_df.xlsx'
path_duk = save_dir + file_prefix + '_0_df.duck'
path_pqt = save_dir + file_prefix + '_0_df.parquet'


list_pdfs = []
for path in Path( test_path ).glob("**/*.pdf"):
  list_pdfs.append( str(path) )
  list_pdfs.sort()


Success!!!


### DASK

Primero creo un cluster locar de computación

In [2]:
from dask.distributed import LocalCluster
client = LocalCluster().get_client()

Luego, con el listado de OT's de ejecutado en primera instancia, genero objetos en los cluster de computación local, 

Ejecuto la función `load_ot()` y guardo los resultados a la misma lista de objetos

In [3]:

futures = [client.submit(gestionOT.GestionOt, file, actor=True) for file in list_pdfs ]
ot_array = [future.result() for future in futures]

ot_cargada = [ot.load_ot() for ot in ot_array]
obj_lists = [future.result() for future in ot_cargada]



Success!!!


🔥

TODO:  Subir a Mongo DB || Mongo Driver

🔥
> TODO
> Imprimir un reporte de las OT que no fue exitoso su conversion a OT. informar las novedades encontradas

🔥
> TODO 
> La siguiente linea de código es posible que no se este ejecutando en paralelo, verificarlo luego

Estoy simultaneamente, generando `matriz` que es un `Pandas.Dataframe` almacenandola en el objeto y en `ot_matrices`

In [4]:
ot_matrices = [ matrizActividades.ConvertirOT_a_ActividadesCSV(ot) for ot in obj_lists ]
df_total = [ df for df in ot_matrices if df is not None ]

In [13]:
# Combined full df for exporting to excel

combined_df = pd.concat(df_total, ignore_index=True)

In [7]:

with open( path_obj, 'wb' ) as fp:
  pickle.dump( obj_lists, fp )

combined_df.to_pickle( path_pkl )

In [10]:
import duckdb

# create the table "my_table" from the DataFrame "my_df"
# Note: duckdb.sql connects to the default in-memory database connection
duckdb.sql("CREATE TABLE duck AS SELECT * FROM combined_df")

# insert into the table "my_table" from the DataFrame "my_df"
duckdb.sql("INSERT INTO duck SELECT * FROM combined_df")

## EXCELL Export

Primero importar el dataframe

Segundo Pintar el dataframe

Agrupar por mes

Exportar

POR HACER



In [8]:

df = pd.read_pickle( path_pkl )

df['Cuadrilla'] = df['Cuadrilla'].apply( lambda s: s.split('(')[0] )

df['Cuenta']         = pd.Categorical(df.Cuenta)
df['Dia']            = pd.Categorical(df.Dia)
df['Alimentador']    = pd.Categorical(df.Alimentador)
df['Tipo']           = pd.Categorical(df.Tipo)
df['Actividad']      = pd.Categorical(df.Actividad)
df['Cuadrilla']      = pd.Categorical(df.Cuadrilla)
df['Responsable']    = pd.Categorical(df.Responsable)
df['Vehiculo']       = pd.Categorical(df.Vehiculo)
df['anio_mes']       = df['Fecha'].apply( lambda x: x[:-3] )
df['anio_mes']       = pd.Categorical(df.anio_mes)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2671 entries, 0 to 2670
Data columns (total 24 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   Item           2671 non-null   int64   
 1   Cuenta         2671 non-null   category
 2   Evento         2671 non-null   object  
 3   Actividad      2671 non-null   category
 4   Alimentador    2671 non-null   category
 5   Primario       2671 non-null   object  
 6   Desconexion    2671 non-null   object  
 7   SIG            2671 non-null   object  
 8   Tipo           2671 non-null   category
 9   Materiales     2671 non-null   object  
 10  Cuadrilla      2671 non-null   category
 11  Dia            2671 non-null   category
 12  Fecha          2671 non-null   object  
 13  InicioEvento   2671 non-null   object  
 14  FinEvento      2671 non-null   object  
 15  Responsable    2671 non-null   category
 16  Colaboradores  2671 non-null   int64   
 17  HorasExtra     2671 non-null   ob

In [14]:
df['Actividad'].unique()

['INFO', 'TRANSP', 'NO PROG', 'LABORA', 'PROG', 'ALIMEN', '·']
Categories (7, object): ['ALIMEN', 'INFO', 'LABORA', 'NO PROG', 'PROG', 'TRANSP', '·']

In [10]:
import xlsxwriter

# Create an ExcelWriter object
writer = pd.ExcelWriter( path_xls, engine='xlsxwriter')  


# Group the DataFrame by the categorical column
grouped = df.groupby('Cuadrilla')

# Iterate through groups and write to separate sheets
for category, group_data in grouped:
    group_data.to_excel(writer, sheet_name=str(category), index=False)  

# Save the Excel file
writer.close()


/tmp/ipykernel_69177/1222790251.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby('Cuadrilla')


# TESTING

In [12]:
df_total[1][['Item','Cuenta','Evento',     'Responsable', 'Colaboradores',  ]]

,Item,Cuenta,Evento,Responsable,Colaboradores
0,1,informativa,En la agencia de la EERSSA El Pangui se coordi...,MORALES RIVERA LUIS ALBERTO,3
1,2,Redes,"Pangui en el centro de la ciudad, calle Quito ...",MORALES RIVERA LUIS ALBERTO,3
3,4,Alumbrado,RECLAMO No. 1100629005 03-07-24/12:05. Pangui ...,MORALES RIVERA LUIS ALBERTO,3
6,7,Alumbrado,RECLAMO No. 1100629005 03-07-24/12:05. Pangui ...,MORALES RIVERA LUIS ALBERTO,3
9,10,transporte,Nos trasladamos desde el Pangui hacia Pachicutza.,MORALES RIVERA LUIS ALBERTO,3
10,11,Alumbrado,En el sector de Pachicutza junto al parterre d...,MORALES RIVERA LUIS ALBERTO,3
11,12,transporte,Nos trasladamos desde el sector de Pachicutza ...,MORALES RIVERA LUIS ALBERTO,3
12,13,Alumbrado,"En el sector El Padmi, en la estructura. No. 9...",MORALES RIVERA LUIS ALBERTO,3
13,14,Alumbrado,En el sector de Los Encuentros estructura. No....,MORALES RIVERA LUIS ALBERTO,3
14,15,lunch,Lunch el sector de Los Encuentros.,MORALES RIVERA LUIS ALBERTO,3


In [6]:
[print( f" Ot Link: \"{ot.link} \" | {ot.log} ") for ot in obj_lists]

 Ot Link: "/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/10_Test Orden de trabajo Zamora 11-02-2022 (RM - Electricistas).pdf " | [{'t': '2025-02-20T23:10:20.222152', 'level': 'INFO', 'message': 'Se encuentra un archivo PDF de al menos tres hojas ', 'detail': 'Ninguno'}, {'t': '2025-02-20T23:10:29.165016', 'level': 'INFO', 'message': 'Desde >> Obtener fechaModa. No se encontro fecha en las actividades', 'detail': 'Se utiliza como fechaModa la fecha de Inicio en la Hoja 1'}] 
 Ot Link: "/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/11_LM_Tres_hojas.pdf " | [{'t': '2025-02-20T23:10:20.234353', 'level': 'INFO', 'message': 'Se encuentra un archivo PDF de al menos tres hojas ', 'detail': 'Ninguno'}] 
 Ot Link: "/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/11_accidente canoa SI.pdf " | [{'t': '2025-02-20T23:10:20.234476', 'level': 'INFO', 'message': 'Se encuentra un archivo PDF de al menos tres hojas ', 'detail': 'Ninguno'}] 
 Ot Link: "/home/vlad/GIT/eers

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [ ]:
[print( file ) for file in list_pdfs]

In [ ]:
[print( f" Ot Link: {ot.log} ") for ot in obj_lists]

### Generar la Matriz de Actividades para un objeto

In [ ]:
nro_ot = 0
obj_lists[nro_ot].link

'/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/find_bug/10_Test Orden de trabajo Zamora 11-02-2022 (RM - Electricistas).pdf'

In [6]:
test = obj_lists[nro_ot].load_ot()
actividades = pd.DataFrame(test.data["actividades"])

In [7]:
fechaModa = test.data['fecha']
type(fechaModa)

str

In [15]:
test.data['log']

[{'t': '2025-03-24T22:43:22.499520',
  'level': 'INFO',
  'message': 'Se encuentra un archivo PDF de al menos tres hojas ',
  'detail': 'Ninguno'},
 {'t': '2025-03-24T22:43:22.626795',
  'level': 'ERROR',
  'message': 'No se ha podido convertir la Fecha Final HOJA DOS a DATE-TIME',
  'detail': '|>> Desde la funcion Linea 383 gestionOT.py <<|'},
 {'t': '2025-03-24T22:43:22.626808',
  'level': 'ERROR',
  'message': 'No se ha podido convertir la Fecha Final HOJA DOS a DATE-TIME',
  'detail': "|>> Desde la funcion 'toDateEcuador()' <<|"},
 {'t': '2025-03-24T22:43:25.698058',
  'level': 'INFO',
  'message': 'Desde >> Obtener fechaModa. No se encontro fecha en las actividades',
  'detail': 'Se utiliza como fechaModa la fecha de Inicio en la Hoja 1'},
 {'t': '2025-03-24T22:43:59.246373',
  'level': 'INFO',
  'message': 'Se encuentra un archivo PDF de al menos tres hojas ',
  'detail': 'Ninguno'},
 {'t': '2025-03-24T22:43:59.365595',
  'level': 'ERROR',
  'message': 'No se ha podido convertir 

In [18]:
matriz_test = matrizActividades.ConvertirOT_a_ActividadesCSV(  obj_lists[nro_ot] )
#matriz_test[['Cuenta','Evento','Fecha','InicioEvento','FinEvento']]
#matriz_test[['Fecha','InicioEvento','corregir_fechaInicio','FinEvento','corregir_fechaFin']]
matriz_test[['Fecha','InicioEvento','FinEvento']]

/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MultinomialNB from version 1.4.1.post1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator CountVectorizer from version 1.4.1.post1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,Fecha,InicioEvento,FinEvento
0,2024-05-22 00:00:00,2024-05-22 08:00:00,2024-05-22 08:22:00
2,2024-05-22 00:00:00,2024-05-23 08:22:00,2024-05-23 08:33:00
3,2024-05-22 00:00:00,2024-05-22 08:53:00,2024-05-22 09:28:00
5,2024-05-22 00:00:00,2024-05-22 09:28:00,2024-05-22 09:59:00
7,2024-05-22 00:00:00,2024-05-22 09:59:00,2024-05-22 10:08:00
8,2024-05-22 00:00:00,2024-05-22 10:08:00,2024-05-22 10:39:00
9,2024-05-22 00:00:00,2024-05-22 10:39:00,2024-05-22 11:35:00
10,2024-05-22 00:00:00,2024-05-22 11:35:00,2024-05-22 12:30:00
11,2024-05-22 00:00:00,2024-05-22 12:30:00,2024-05-22 13:30:00
12,2024-05-22 00:00:00,2024-05-22 13:30:00,2024-05-22 14:18:00


In [15]:
dbg

version                                                     0.12.0
link             /home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/te...
id_ot                                                     134508.0
exito                                                         True
cuadrilla                          Yacuambi Z1 (Cuadrilla. Nro. 8)
responsable                     [LOZANO SIGCHO NAUN ENRIQUE, JECE]
colaboradores    {'total': 3, 'nombres': [['CABRERA GONZALEZ LU...
diaSemana                                                    lunes
fecha                                    2024-07-29 00:00:00-05:00
fechaInicio                            lunes, 29 de julio del 2024
fechaFinal                                     29/07/2024 20:40:00
sitio                 Yacuambi - Tamboloma, Hucapamba y Jembuentza
descripcion      Traslado a Tamboloma para revisar sector sin s...
tEstimado                                                        8
vehiculo         {'numero': 'R-171', 'placa': 'AAA-4278', 'mar

In [16]:
test.data['fechaFinal']

'11/02/2022 23:00:00'

In [26]:
from datetime import datetime
from pytz import timezone

fechaFinal = test.data['fechaFinal']

datetime_object = datetime.strptime(fechaFinal, '%d/%m/%Y %H:%M:%S')
ecuador = timezone("America/Guayaquil")
local_datetime = ecuador.localize(datetime_object)
fechaFinal = local_datetime.isoformat()
solofechaFinal = fechaFinal.split('T')[1]
solofechaFinal

'23:00:00-05:00'

### Secuencial GLOBAL LOCK

In [ ]:
### Secuencial en un solo procesador.
obj_lists = []
for file in list_pdfs:
  ot = gestionOT.GestionOt( file )
  ot.load_ot()
  obj_lists.append( ot )